# Chapter 7: Direct Preference Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part2_core/07_dpo.ipynb)

> **Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar & Michael Chertushkin (Packt, 2025)  
> **Chapter 7:** Direct Preference Optimization  
> **Notebook:** `part2_core/07_dpo.ipynb`

---

## What this notebook covers

This is the companion notebook for Chapter 7 of the book. Run it on a free Colab T4 GPU. All code uses small, publicly available models (under 500 MB) that fit within the free tier memory limit.


### Before we start

In the previous part we discussed LLM training phases and performed **full-precision Supervised Finetuning** on a dataset of `(question, answer)` pairs, teaching an LLM to generate citations in the end of its response.

In this notebook we will try to achieve the same functionality. But instead of SFT, we'll try **Direct Preference Optimization** (**DPO**).


### What is DPO

**Direct Preference Optimization** (**DPO**) was first suggested for the Human Alignment task as an alternative to the quite finicky Reinforcement Learning procedure based on PPO (Proximal Policy Optimization) algorithm.

Though DPO was invented by analytically finding the maximum of the RLHF loss, it has nothing to do with RL. It's just Supervised Finetuning on a dataset of triplets `(prompt, accepted_completion, rejected_completion)` with a special loss function that stimulates the LLM to prioritize accepted completions and pessimize unwanted ones.

For a while, DPO was quite popular as an algorithm for establishing Human Alignment. But for now (July 2025), RL tactics **PPO** (the algorithm behind the original **RLHF**) and **GRPO** (use to train **DeepSeek-R1** for long reasoning) are still considered the mainstream types of post-training. This is especially true outside Human Alignment, in the tasks where completions are scored according to the "correctness" of sorts - might be answer correction or web scenario goal achievement.

Learn more about DPO in the relevant chapters of the Book.


### How is DPO different from SFT?

Previously (in 2023 and 2024), DPO and SFT were in different phases. SFT was on the phase of post-training and DPO was usually applied after SFT during phase of Human Alignment. Today the line between post-training and human alignment became more blurry and DPO and SFT can be applied even in parallel. Let's understand some functional differences between the two:

- While Supervised Fine-Tuning (SFT) teaches a model what to say, Direct Preference Optimization (DPO) teaches it what to prefer.
- SFT treats all demonstrations as ground truth, while DPO contrasts preferred and less preferred outputs to learn a more nuanced decision boundary.
- Unlike SFT, which optimizes the likelihood of single responses, DPO uses relative preference data to optimize over pairs—training the model to assign higher probability to preferred outputs
- DPO achieves similar goals to PPO-based RLHF, but in a fully supervised and differentiable way, avoiding reward models and policy rollouts.


### Ok, but how can I implement DPO?

In order to implement DPO you can manually implement the loss function (and this will be your take-home task), but in this notebook we will explore a much simpler alternative from HuggingFace: the `DPOTrainer` class (which has the loss implemented for you)

We'll be working on the same task as in the Chapter 5 notebook - trying to reduce LLM hallucination by forcing it to cite references. But instead of SFT, we'll use DPO.


### Our Objective

We will fine-tune the model such that it always generates the list of citations in the end in the specific format.

#### Example
Before DPO:
```
Q: What is the purpose of Norepinephrine?
A: Norepinephrine is a neurotransmitter that is released from the sympathetic nervous system, and it is involved in the regulation of the body's response to stress.
```

After DPO:
```
Q: What is the purpose of Norepinephrine?
A: Norepinephrine is a neurotransmitter [260104] that is released from the sympathetic nervous system, and it is involved in the regulation of the body's response to stress [283012].

References:
[260104] The role of noradrenaline and adrenaline in the nervous circuitry ...
[283012] Stress-induced reactions of the body provoke increase in levels of ...
```


### Dataset preparation

For DPO, we'll need the dataset in the form `(prompt, chosen_completion, rejected_completion)`. More accurately, the `DPOTrainer` expects data as follows:

```python
data = [
    {
        "prompt": "Why is the sky blue?",
        "chosen": "Because the atmosphere scatters blue light.",
        "rejected": "Because it's the color of the ocean reflected back."
    },
    {
        "prompt": "What is the capital of France?",
        "chosen": "The capital of France is Paris.",
        "rejected": "France has no capital."
    }
]
```

Since we're solving the LLM grounding task, the `chosen` completion should be with references cited and the `rejected` completion should cite no references. Like this:

```python
    {
        "prompt":
            """Question: Are long non coding RNAs spliced?
            Context:
            Document # 22955974:
            Splicing remains an incompletely understood process...
            Document # 22707570:
            Thousands of long noncoding RNAs (lncRNAs) have been found....
            """,
        "chosen": """Long non coding RNAs appear to be spliced through the same pathway as the mRNAs [22955988, 24285305].

            References:
            [22955988] The human genome contains many thousands...
            [24285305] NONCODE (http://www.bioinfo.org/noncode/) is an...""",
        "rejected": "Long non coding RNAs appear to be spliced through the same pathway as the mRNAs"
    },

```


In [ ]:
# Updating gcsfs along with fsspec to resolve dependency conflicts
!pip install -q transformers trl fsspec==2023.9.2 gcsfs==2023.9.2 datasets==3.0.0

Here's the actual dataset we'll be fine tuning our LLM on:


In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/arunpshankar/packt-final/main/code/data/samples/with_sft.csv')
df

In [ ]:
print(df['prompt'].values[0])

In [ ]:
print(df['sft'].values[0])

#### Dataset format explanation

- `df['prompt']` contain text in the format:

  ```Question: {some_question}\n Context: {some_context}\nAnswer:```

  Yes, not only prompts, but also context and answers. Before training, we'll get rid of the answers :)

- `df['sft']` contains the text with the format:

  ```{Some_answer}\n References: {some_references}```
  
  This is our actual target.


Let's modify the prompt such that we always have 3 values:
- Prompt
- Chosen (answer with citations)
- Rejected (answer without citations)


In [ ]:
print(df['prompt'].values[1]) # this is the prompt + answer

In [ ]:
print(df['sft'].values[1]) # this is what we want

In [ ]:
print(df['prompt'].apply(lambda x: x.split('Answer:')[1].strip()).values[1]) # and this is the standard output without citations

In [ ]:
from transformers import AutoTokenizer

model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)

df['chosen'] = df['sft']
df['rejected'] = df['prompt'].apply(lambda x: x.split('Answer:')[1].strip())

# Wrap user prompts in the model's chat template so the prompt/response
# boundary tokenizes identically standalone vs. concatenated with chosen/rejected.
# This is what silences the DPOTrainer "Mismatch between tokenized prompt..." warnings.
_raw_prompts = df['prompt'].apply(lambda x: x.split('Answer:')[0].strip())
df['prompt'] = _raw_prompts.apply(
    lambda p: tokenizer.apply_chat_template(
        [{"role": "user", "content": p}],
        tokenize=False,
        add_generation_prompt=True,
    )
)

In [ ]:
df['overall_max_text'] = df['prompt'] + df['chosen']

#### Analysis of dataset

Let's look at how many words do we have on average in the corpus. This will help us estimate the amount of memory we will need.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
sns.histplot(df['overall_max_text'].apply(lambda x: len(x.split())))
plt.xlabel('Length of Text (Words)')
plt.ylabel('Count')

We see that we have quite a lot of texts that are unfortunately larger than 512 words - these texts will not fit into memory in the free version of Google Colab, so we will filter them out.

**Important Note**

We will filter texts who length is larger than 512 / 1.3 = 393. This is done because 1 word produces on average ~1.3 tokens after BPE tokenization, and max_len=512 is actually a limit on BPE tokens and not on words.


In [ ]:
MAX_LEN = 512
BPE_FACTOR = 1.3

In [ ]:
df = df[df['overall_max_text'].apply(lambda x: len(x.split())) < MAX_LEN / BPE_FACTOR]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
sns.histplot(df['overall_max_text'].apply(lambda x: len(x.split())))
plt.xlabel('Length of Text (Words) After Removing Long Texts')
plt.ylabel('Count')

# We keep the max_len < 512 / 1.3,
# because after we apply tokenizer, each word will generate on average ~1.3 tokens

Let's split the dataset into train and validation part. We don't need large validation part, so let's just take 5% of the original dataset.


In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df[['prompt', 'chosen', 'rejected']], test_size=0.05, random_state=42)
datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df)
})

In [ ]:
datasets

In [ ]:
from trl import DPOConfig, DPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
max_length = int(MAX_LEN * BPE_FACTOR) # because of BPE

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

#### Intrinsic Evaluation

Let's use one of the methods of intrinsic evaluation. For those who are not coming from classical NLP, here is a quick refresher:
- Intrinsic Evaluation is the process of estimating the performance of the model using some proxy/not-directly-linked-to-the-task metrics. As an example of intrinsic evaluation we can get the log porbabilities of chosen and rejected.
- Extrinsic Evaluation on the contrary measures the model "in the field", on the concrete deployed setup using real-word concrete task.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm import tqdm

def get_logprob(completion, prompt):
    full_text = prompt + completion
    inputs = tokenizer(full_text, return_tensors="pt", truncation=True).to("cuda")
    labels = inputs["input_ids"].clone()

    # Mask out the prompt tokens — don't compute loss on them
    prompt_len = len(tokenizer(prompt, return_tensors="pt")["input_ids"][0])
    labels[0, :prompt_len] = -100  # ignore_index for loss

    with torch.no_grad():
        output = model(**inputs, labels=labels)
        return -output.loss.item()  # Higher score = more likely


def get_win_rate(dataset):
    correct = 0
    total = 0

    for example in tqdm(dataset):
        prompt = example["prompt"]
        chosen = example["chosen"]
        rejected = example["rejected"]

        score_chosen = get_logprob(chosen, prompt)
        score_rejected = get_logprob(rejected, prompt)

        if score_chosen > score_rejected:
            correct += 1
        total += 1

    print(f"Win rate: {correct / total:.2%}")

get_win_rate(datasets['validation'])

We see that win rate before finetuning is very low. Let's generate something from the model!


In [ ]:
sample_prompts = df['prompt'].iloc[:5].tolist()
sample_gt = df['chosen'].iloc[:5].tolist()
pretrain_outputs = []
print("=== Before fine-tuning ===")
for i, (p, gt) in enumerate(zip(sample_prompts, sample_gt)):
    print(f'Reference answer #{i}:')
    print(gt)
    print('-----------------------------------')
    print(f'Prediction #{i}:')
    inputs = tokenizer(p, truncation=True, max_length=max_length, return_attention_mask=True, return_tensors="pt").to(model.device)
    out_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True
    )[0]
    prediction = tokenizer.decode(out_ids, skip_special_tokens=True)
    pretrain_outputs.append(prediction)
    print(prediction)
    print("-----------------------------------\n\n\n")

In [ ]:
from IPython.display import HTML, display

table_template = """<table style="border:1px solid black; table-layout: fixed; width: 100%;" >
  <tr>
    <th style="text-align: center; border:1px solid black; width: 33%;">PREFIX</th>
    <th style="text-align: center; border:1px solid black; width: 33%;">PRETRAIN</th>
    <th style="text-align: center; border:1px solid black; width: 33%;">DPO</th>
  </tr>
{}
</table>"""

# Modified row_template for text wrapping and better alignment
row_template = '''  <tr>
    <td style="border:1px solid black; vertical-align:top;"><pre style="white-space: pre-wrap; word-wrap: break-word; text-align:left; margin: 0;">`{}`</pre></td>
    <td style="border:1px solid black; vertical-align:top;"><pre style="white-space: pre-wrap; word-wrap: break-word; text-align:left; margin: 0;">{}</pre></td>
    <td style="border:1px solid black; vertical-align:top;"><pre style="white-space: pre-wrap; word-wrap: break-word; text-align:left; margin: 0;">{}</pre></td>
  </tr>'''


rows = []

for i, prefix in enumerate(sample_prompts):
    rows.append(row_template.format(prefix, pretrain_outputs[i], None))

display(HTML(table_template.format('\n'.join(rows))))

In [ ]:
from trl import DPOConfig, DPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
# Switched deprecated torch_dtype to dtype
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16).to('cuda')
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

training_args = DPOConfig(
    beta=0.1,
    learning_rate=5e-6,
    max_length=768,
    remove_unused_columns=False,
    gradient_accumulation_steps=4,
    output_dir="Qwen2.5-0.5B-Instruct-DPO",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    logging_steps=10
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=datasets['train'],
    eval_dataset=datasets['validation'],
)

In [ ]:
trainer.train()

In [ ]:
get_win_rate(datasets['validation'])

In [ ]:
posttrain_outputs = []
print("=== After fine-tuning ===")
for i, (p, gt) in enumerate(zip(sample_prompts, sample_gt)):
    print(f'Reference answer #{i}:')
    print(gt)
    print('-----------------------------------')
    print(f'Prediction #{i}:')
    inputs = tokenizer(p, truncation=True, max_length=max_length, return_attention_mask=True, return_tensors="pt").to(model.device)
    out_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True
    )[0]
    prediction = tokenizer.decode(out_ids, skip_special_tokens=True)
    posttrain_outputs.append(prediction)
    print(prediction)
    print("-----------------------------------\n\n\n")

In [ ]:
rows = []

for i, prefix in enumerate(sample_prompts):
    prefix = prefix
    rows.append(row_template.format(prefix, pretrain_outputs[i], posttrain_outputs[i]))

display(HTML(table_template.format('\n'.join(rows))))

Let's quickly look one more time at what was `chosen` and what was `rejected`


In [ ]:
df['rejected'].values[2]

In [ ]:
df['chosen'].values[2]

In [ ]:
# trainer.evaluate()

### Conclusion

With the DPO trainer we were able to achieve the same behaviour as we did with SFT Full Precision finetuning. Whether to choose SFT or DPO is really dependendent in your task and dataset. There is a consensus that DPO changes `how` the text is generated whereas SFT in contrary can be used to change `what exactly` will be generated.
